In [7]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import fft, signal
from scipy.io.wavfile import read
from scipy.fft import fft, fftfreq
import glob
from typing import List, Dict, Tuple
import pickle

In [ ]:
def crear_constelacion(audio, Fs):

    ventana_muestreo_longitud = 0.5
    ventana_muestreo_numero = int(ventana_muestreo_longitud * Fs)
    ventana_muestreo_numero += ventana_muestreo_numero % 2
    num_picos = 15

    cantidad_completar = ventana_muestreo_numero - audio.size % ventana_muestreo_numero
    cancion_entrada = np.pad(audio, (0, cantidad_completar))

    frecuencias, tiempos, stft = signal.stft(
        cancion_entrada, Fs, nperseg=ventana_muestreo_numero, nfft=ventana_muestreo_numero, return_onesided=True)
   
    mapa_constelacion = []
    for tiempo_indice, ventana in enumerate(stft.T):
        espectro = abs(ventana)
        picos, props = signal.find_peaks(espectro, prominence=0, distance=200)
        n_picos = min(num_picos, len(picos))
        picos_prominentes = np.argpartition(props["prominences"], -n_picos)[-n_picos:]
        for pico in picos[picos_prominentes]:
            frecuencia = frecuencias[pico]
            mapa_constelacion.append([tiempo_indice, frecuencia])
    return mapa_constelacion

In [ ]:
def crear_hashes(mapa_constelacion, cancion_id=None):
   
    hashes = {}
    frecuencia_superior = 23_000 
    frequencia_bits = 10

    for idx, (tiempo, frec) in enumerate(mapa_constelacion):
        
        for otro_tiempo, otra_frec in mapa_constelacion[idx : idx + 100]: 
            dif_tiempo = otro_tiempo - tiempo
        
            if dif_tiempo <= 1 or dif_tiempo > 10:
                continue
        
            frec_conv = frec / frecuencia_superior * (2 ** frequencia_bits)
            otra_frec_conv = otra_frec / frecuencia_superior * (2 ** frequencia_bits)
            hash = int(frec_conv) | (int(otra_frec_conv) << 10) | (int(dif_tiempo) << 20)
        
            hashes[hash] = (tiempo, cancion_id)
    return hashes

In [ ]:
def puntuacion_canciones(hashes):
    emparejamientos_por_cancion = {}
    for hash, (tiempo_muestreo, _) in hashes.items():
        if hash in base_de_datos:
            emparejamiento = base_de_datos[hash]
            for tiempo_ref, indice_cancion in emparejamiento:
                if indice_cancion not in emparejamientos_por_cancion:
                    emparejamientos_por_cancion[indice_cancion] = []
                emparejamientos_por_cancion[indice_cancion].append((hash, tiempo_muestreo, tiempo_ref))
            
    puntuaciones = {}
    for indice_cancion, matches in emparejamientos_por_cancion.items():
        cancion_puntuaciones_offset = {}
        for hash, tiempo_muestreo, tiempo_ref in matches:
            delta = tiempo_ref - tiempo_muestreo
            if delta not in cancion_puntuaciones_offset:
                cancion_puntuaciones_offset[delta] = 0
            cancion_puntuaciones_offset[delta] += 1

        max = (0, 0)
        for offset, score in cancion_puntuaciones_offset.items():
            if score > max[1]:
                max = (offset, score)
        
        puntuaciones[indice_cancion] = max

    puntuaciones = list(sorted(puntuaciones.items(), key=lambda x: x[1][1], reverse=True)) 
    
    return puntuaciones

In [ ]:
songs = glob.glob('/content/drive/MyDrive/Shazam/data/*.wav') #Ingrese ruta de la carpeta de la base de datos con canciones
cancion_indice = {}
base_de_datos: Dict[int, List[Tuple[int, int]]] = {}

for indice, archivo in enumerate(sorted(songs)):
    cancion_indice[indice] = archivo
    Fs, audio_entrada = read(archivo)
    audio_entrada = audio_entrada[:,0]
    constelacion = crear_constelacion(audio_entrada, Fs)
    hashes = crear_hashes(constelacion, indice)

    for hash, par_indice_tiempo in hashes.items():
        if hash not in base_de_datos:
            base_de_datos[hash] = []
        base_de_datos[hash].append(par_indice_tiempo)

with open("base_de_datos.pickle", 'wb') as db:
    pickle.dump(base_de_datos, db, pickle.HIGHEST_PROTOCOL)
with open("cancion_indice.pickle", 'wb') as songs:
    pickle.dump(cancion_indice, songs, pickle.HIGHEST_PROTOCOL)


In [ ]:
base_de_datos = pickle.load(open('base_de_datos.pickle', 'rb'))
direccion_cancion = pickle.load(open("cancion_indice.pickle", "rb"))
Fs, audio_entrada = read("/content/drive/MyDrive/Shazam/pruebas/prueba1.wav") #Inserte ruta con archivo de audio de la canción que desee igentificar
audio_entrada = audio_entrada[:,0]
constelacion = crear_constelacion(audio_entrada, Fs)
hashes = crear_hashes(constelacion, None)
puntuaciones = puntuacion_canciones(hashes)

for cancion_indice, puntuacion in puntuaciones:
    print(f"{direccion_cancion[cancion_indice]}=: Score of {puntuacion[1]} at {puntuacion[0]}")

/content/drive/MyDrive/Shazam/data/partita1preludio.wav=: Score of 976 at 168
/content/drive/MyDrive/Shazam/data/partita1gigue.wav=: Score of 242 at 169
/content/drive/MyDrive/Shazam/data/partita1allemande.wav=: Score of 231 at 360
/content/drive/MyDrive/Shazam/data/partita1sarabande.wav=: Score of 211 at 660
/content/drive/MyDrive/Shazam/data/partita1corrente.wav=: Score of 186 at 357
/content/drive/MyDrive/Shazam/data/partita1minuetos.wav=: Score of 117 at 236
